# KKBox — Behavioral Features

**Purpose:** turn the 410 million daily listening rows into features for each user and target month, using only data on or before the prediction cutoff (the last day of the previous month), as designed in [docs/prediction_design.md](../docs/prediction_design.md).

**Evidence labels:** every result is a full-data result for the stated month. Statements marked *hypothesis* are not verified. Missing listening is **not** zero listening: users without logs get an explicit flag.

**Reading order:** setup → check of the log windows on the February 2017 population → (next) listening features → subscription-history features → leakage check → all months.

**Run context:** open from the `notebooks/` directory. The user-log Parquet files are on an external drive configured in `local_settings.py` (git-ignored); connect the drive first. The February 2017 population comes from the cache written by notebook 02 (`data/processed/`).

### 0. Setup

Registers the user-log Parquet file as a DuckDB view (nothing is loaded into memory) and loads the rebuilt February 2017 population (users whose last expiration at the 2017-01-31 cutoff lies in February 2017) from notebook 02's cache.

**What to look for:** no error, and 879,537 users.

In [1]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src import audit_checks as ac
from src.audit_display import show

CACHE_DIR = Path.cwd().parent / "data" / "processed"
from local_settings import PARQUET_DIR, TEMP_DIR   # machine-specific, git-ignored

con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute(f"SET temp_directory='{TEMP_DIR}'")

ac.register_parquet(con, "logs_v1", Path(PARQUET_DIR) / "user_logs.parquet")

pop = pd.read_parquet(CACHE_DIR / "rebuilt_monthly_labels.parquet", columns=["target_month", "msno"])
pop = pop[pop.target_month == "2017-02"][["msno"]]
con.register("pop", pop)
show("February 2017 population", pd.DataFrame({"users": [len(pop)]}))

**February 2017 population**

,users
0,"879,537"


### 1. Do the log windows reproduce the earlier figures?

**Purpose:** before building features, check that the window definitions and the cutoff are right by reproducing the exploratory figures in `docs/prediction_design.md`: for the February 2017 population, 68.8% have a log in the last 7 days before the cutoff, 77.1% in 30 days, 80.5% in 90 days.

**Logic:** the cutoff is 2017-01-31. Each window counts the cutoff day and the days before it (7 days from 2017-01-25, 30 days from 2017-01-02, 90 days from 2016-11-03). Dates are text in `YYYYMMDD` form, so text comparison orders them correctly. Users without any log in the window are kept (left join).

**What to look for:** three percentages close to 68.8, 77.1, and 80.5. A clear difference means a window start or the cutoff is off; check those before suspecting the data. This scans about 390 million rows and can take a few minutes.

In [2]:
sql = """
WITH per_user AS (
    SELECT p.msno,
           count(*) FILTER (WHERE l.date >= '20170125') AS days_7,
           count(*) FILTER (WHERE l.date >= '20170102') AS days_30,
           count(l.date)                                AS days_90
    FROM pop p
    LEFT JOIN logs_v1 l ON l.msno = p.msno AND l.date >= '20161103' AND l.date <= '20170131'
    GROUP BY p.msno)
SELECT count(*) AS users,
       100.0 * avg((days_7  > 0)::int) AS log_in_last_7_days_percentage,
       100.0 * avg((days_30 > 0)::int) AS log_in_last_30_days_percentage,
       100.0 * avg((days_90 > 0)::int) AS log_in_last_90_days_percentage
FROM per_user
"""
show("Users with a listening log before the 2017-01-31 cutoff", con.sql(sql).df(),
     note="Windows count the cutoff day and the days before it; nothing after the cutoff is used.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Users with a listening log before the 2017-01-31 cutoff**  
*Windows count the cutoff day and the days before it; nothing after the cutoff is used.*

,users,log_in_last_7_days_percentage,log_in_last_30_days_percentage,log_in_last_90_days_percentage
0,"879,537",68.85%,77.11%,80.48%


### 2. Listening features for February 2017

**Purpose:** build the listening features for one month before scaling to all months. The logic lives in `src/behavior_features.py` (tested), so the same code runs for every cutoff later.

**Logic:** for each user of the February 2017 population, using only log rows dated on or before the 2017-01-31 cutoff: active days, plays, valid listening seconds, unique songs, and the share of completed plays in windows of 7, 14, 30, 60, and 90 days; days since the last log; two trends (the last 7 days against the previous 23, the last 30 against the previous 30, as daily rates). Users with no log in 90 days get a flag and **missing** values, not zeros. `total_secs` outside 0 to 86,400 is treated as missing and flagged (DQ-001); the source is not changed.

**What to look for:** one row per user (879,537); the share without a log in 90 days (about 19.5% from section 1); the distributions of the 30-day features among users with logs; the small share of users with an invalid-seconds flag.

In [ ]:
from src import behavior_features as bf

features_feb = bf.listening_features(con, "logs_v1", "pop", "20170131")

show("February 2017 feature table", pd.DataFrame({
    "users": [len(features_feb)],
    "features": [features_feb.shape[1] - 1],
    "no_log_90_days_percentage": [100 * features_feb["no_log_90_flag"].mean()],
    "invalid_seconds_flag_percentage": [100 * features_feb["has_invalid_secs_90"].mean()]}))
show("30-day listening features among users with a log in 90 days",
     features_feb.loc[features_feb.has_log_90, ["active_days_30", "plays_30", "secs_30", "unique_songs_30",
                                                "completed_share_30", "days_since_last_log"]].describe().T,
     note="Missing listening is not zero: users without a log in 90 days are excluded here and flagged in the table.")

### 3. A first look: churn label rate by listening days (descriptive)

**Purpose:** a sanity check that the features carry signal in the direction found earlier, before any model: the rebuilt churn label rate for February 2017 by the number of listening days in the last 30 days.

**What to look for:** the earlier exploratory rates were 2.76% (no log in 30 days), 6.10% (1–5 days), 5.55% (6–15), 3.80% (16–25), and 2.36% (26–30). Users without any log in 90 days are counted in the "none" band here. This is descriptive for one month, **not** a causal claim, and the low rate among users with no listening is a surprise that needs its own explanation (*hypothesis*: a different user type, for example users who stream elsewhere or share an account).

In [ ]:
labels = pd.read_parquet(CACHE_DIR / "rebuilt_monthly_labels.parquet", columns=["target_month", "msno", "is_churn"])
labels = labels[labels.target_month == "2017-02"][["msno", "is_churn"]]
df = features_feb.merge(labels, on="msno")
df["listening_days_30"] = pd.cut(df["active_days_30"].fillna(0), [-1, 0, 5, 15, 25, 30],
                                 labels=["none", "1-5", "6-15", "16-25", "26-30"])
by_band = df.groupby("listening_days_30", observed=True).agg(
    users=("msno", "size"), churn_label_percentage=("is_churn", lambda s: 100 * s.mean()))
show("Churn label rate by listening days in the last 30 days (February 2017, rebuilt labels)", by_band)

### 4. Findings so far (February 2017, full population)

- **The log windows and the cutoff are right.** 68.85%, 77.11% and 80.48% of the 879,537 users have a log in the last 7, 30 and 90 days before 2017-01-31, matching the figures in `docs/prediction_design.md`.
- **19.52% (171,705 users) have no log in 90 days** and are flagged with missing, not zero, values; 0.46% of users have at least one invalid `total_secs` day in the window (flagged, set to missing).
- **Among users with a log (707,832), listening is spread widely:** median 17 listening days in 30, median 406 plays, and 75% of users with plays complete 63% to 87% of them; 29,609 of these users have no listening in the last 30 days (their completed share is undefined).
- **The churn label rate falls with listening days, but the "no listening" group is low:** 2.76% for none (201,314 users), 6.10% for 1–5 days, 5.55% for 6–15, 3.80% for 16–25, and 2.36% for 26–30 days, identical to the earlier exploratory figures. This is descriptive for one month and not causal. Why users with no listening churn least is open (*hypothesis*: a different user type); to be examined by splitting this group by plan, auto-renew, and tenure.

### 5. Subscription-history and profile features for February 2017

**Purpose:** add the second half of the features: what the user's own subscription record says at the cutoff (tenure, transaction counts, cancellations, the last plan, auto-renew, price paid, days to expiration) and the profile attributes known by then. The logic lives in `src/subscription_features.py` (tested).

**Logic:** transactions up to 2017-01-31 only; the user's *last row* follows the official labeller's ordering, so `days_to_expiration` is the lead time to the last expiration. Tenure is measured from the first transaction in the data and is **left-censored** (the data starts on 2015-01-01), so first transactions in January 2015 are flagged. Profile: age kept only when 1 to 100, gender "unknown" kept, registration date used only when on or before the cutoff. `has_member_profile` is kept for diagnostics and is not a feature.

**Setup (this cell):** registers the raw transaction and member files in DuckDB and a view `tx_all` that stacks the two transaction releases.

In [ ]:
from src import subscription_features as sf

RAW_DIR = Path.cwd().parent / "data" / "raw"
for name, filename in {"tx_v1": "transactions.csv", "tx_v2": "transactions_v2.csv", "members": "members_v3.csv"}.items():
    ac.register_csv(con, name, RAW_DIR / filename)
TX_COLUMNS = ("msno, payment_method_id, payment_plan_days, plan_list_price, actual_amount_paid, "
              "is_auto_renew, transaction_date, membership_expire_date, is_cancel")
con.execute(f"CREATE OR REPLACE VIEW tx_all AS SELECT {TX_COLUMNS} FROM tx_v1 UNION ALL SELECT {TX_COLUMNS} FROM tx_v2")

sub_feb = sf.subscription_features(con, "tx_all", "pop", "20170131")
prof_feb = sf.profile_features(con, "members", "pop", "20170131")
show("Subscription and profile feature tables, February 2017", pd.DataFrame({
    "users": [len(sub_feb)], "subscription_features": [sub_feb.shape[1] - 1], "profile_features": [prof_feb.shape[1] - 1],
    "with_transaction_history_percentage": [100 * sub_feb["has_history"].mean()],
    "with_member_profile_percentage": [100 * prof_feb["has_member_profile"].mean()]}))

**Check against the earlier figures:** `docs/prediction_design.md` records, for this cohort, a median of 15 transactions, 8.9% with a first transaction under 90 days before the cutoff, and 7.7% with at most two transactions (exploratory). The table also shows the lead time to the last expiration by band.

**What to look for:** the three checked figures close to 15, 8.9% and 7.7%; every user has a history (100%, since the population is built from transactions); lead-time bands that cover 1 to 31 days.

In [ ]:
checks = pd.DataFrame({
    "median_transactions": [sub_feb["n_transactions"].median()],
    "first_transaction_under_90_days_percentage": [100 * (sub_feb["tenure_days"] < 90).mean()],
    "at_most_two_transactions_percentage": [100 * (sub_feb["n_transactions"] <= 2).mean()],
    "first_transaction_in_first_data_month_percentage": [100 * sub_feb["first_tx_in_first_data_month"].mean()]})
show("Subscription history checks (February 2017)", checks)

lead = pd.cut(sub_feb["days_to_expiration"], [0, 7, 14, 21, 31], labels=["1-7", "8-14", "15-21", "22-31"])
show("Lead time from the cutoff to the last expiration", lead.value_counts(dropna=False).sort_index()
     .rename("users").to_frame().assign(percentage=lambda t: 100 * t.users / t.users.sum()))

### 6. Churn label rate by subscription feature (descriptive)

**Purpose:** a first look at which subscription features separate churners, before any model: the rebuilt churn label rate (February 2017) by the last plan's auto-renew flag, plan length, price paid, number of transactions, recent cancellations, and lead time.

**What to look for:** large differences between groups, and group sizes (a striking rate in a tiny group means little). One month and descriptive only; these differences are not causal, and early renewers are labeled differently by the official file (open question).

In [ ]:
labels = pd.read_parquet(CACHE_DIR / "rebuilt_monthly_labels.parquet", columns=["target_month", "msno", "is_churn"])
labels = labels[labels.target_month == "2017-02"][["msno", "is_churn"]]
sub = sub_feb.merge(labels, on="msno")

groups = {
    "Last transaction auto-renew": sub["last_auto_renew"].map({1: "on", 0: "off"}),
    "Last plan length": pd.cut(sub["last_plan_days"], [-1, 0, 29, 30, 10000], labels=["0 days", "1-29 days", "30 days", "over 30 days"]),
    "Last payment": sub["last_paid_zero"].map({True: "zero paid", False: "paid"}),
    "Transactions so far": pd.cut(sub["n_transactions"], [0, 2, 6, 12, 24, 1000], labels=["1-2", "3-6", "7-12", "13-24", "25+"]),
    "Cancellations in the last 90 days": pd.cut(sub["n_cancellations_90"], [-1, 0, 1000], labels=["none", "1 or more"]),
    "Lead time (days to expiration)": pd.cut(sub["days_to_expiration"], [0, 7, 14, 21, 31], labels=["1-7", "8-14", "15-21", "22-31"]),
}
for title, group in groups.items():
    table = sub.groupby(group, observed=True).agg(users=("msno", "size"),
                                                  churn_label_percentage=("is_churn", lambda s: 100 * s.mean()))
    table.index.name = None
    show(f"Churn label rate by: {title.lower()} (February 2017)", table)

### 7. Findings: subscription features (February 2017, rebuilt labels, descriptive)

- **The features reproduce the earlier checks:** median 15 transactions, 8.89% with a first transaction under 90 days before the cutoff, 7.70% with at most two transactions. Lead times are spread evenly (22.77%, 22.10%, 23.00% for 1–7, 8–14 and 15–21 days; 32.14% for 22–31, which in February means 22–28 days). **27.52% of users have their first transaction in January 2015**, so for them tenure is truncated by the start of the data; tenure must be read with that flag.
- **Auto-renew is the strongest single signal:** the churn label rate is 21.39% when the last transaction has auto-renew off (104,532 users) against 1.60% when it is on (775,005 users), against 3.95% overall. Users with auto-renew off are about 12% of the population and hold roughly two thirds of the churners (about 64%, computed from the rounded rates). This simple rule is therefore the baseline a model has to beat.
- **Other differences point the same way:** the rate falls steadily with the number of transactions (15.01% for 1–2, 6.53% for 3–6, 4.12% for 7–12, 2.35% for 13–24, 1.19% for 25+); 1 or more cancellations in the last 90 days 9.36% against 3.78% for none; plans over 30 days 27.06% (20,740 users), plans under 30 days 48.73% (only 1,849 users), a zero last payment 39.19% (only 4,021 users).
- **Lead time hardly separates churners** (4.18%, 4.09%, 3.98%, 3.68% from the shortest to the longest band): it matters for how much time an intervention has, not for who churns.
- **Limits:** one month, descriptive, not causal; the features overlap (for example auto-renew and plan length), so the single-feature rates cannot be added up; small groups (under 30 days, zero paid) are noisy; early renewers are labeled differently by the official file (open question).